# Álgebra Lineal — Módulo 01: Sistemas de Ecuaciones Lineales, Eliminación Gauss-Jordan y Rouché-Frobenius 3D

> **Institución:** Universidad San Sebastián (USS) — Sede Patagonia  
> **Carrera:** Ingeniería Civil Informática  
> **Asignatura:** Álgebra Lineal  
> **Docente:** Carol Asencio González  
> **Entorno:** Python 3, SymPy, NumPy, Matplotlib 3D, ipywidgets

---

### 🏷️ Leyenda de Trazabilidad de Fuentes
- 🎓 `[Cátedra USS / Diapositivas Docente]`: Operaciones Elementales por Fila (OEF), Forma Escalonada Reducida por Filas (RREF) y Teorema de Rouché-Frobenius.
- 📖 `[Texto Guía — Stanley Grossman / Poole]`: Interpretación geométrica de hiperplanos en $\mathbb{R}^3$, caracterización de grados de libertad y unicidad de soluciones.
- 🌐 `[Enriquecimiento Web / Computación Científica]`: Cálculo simbólico exacto con SymPy, visualización de planos volumétricos y paneles interactivos con `ipywidgets`.

---

> [!NOTE]
> Un sistema de ecuaciones lineales $A\mathbf{x} = \mathbf{b}$ de orden $3 \times 3$ representa algebraicamente la intersección de tres planos en el espacio euclidiano tridimensional $\mathbb{R}^3$. El Teorema de Rouché-Frobenius permite diagnosticar rigurosamente la existencia y multiplicidad de soluciones a través de la comparación de rangos entre la matriz de coeficientes $A$ y la matriz ampliada $(A \mid \mathbf{b})$.

## 1. Fundamentos Teóricos: Álgebra y Geometría en $\mathbb{R}^3$

### 1.1 Representación Matricial y Operaciones Elementales por Fila (OEF)
Dado el sistema de $3$ ecuaciones con $3$ incógnitas:

$$
\begin{cases}
a_{11}x + a_{12}y + a_{13}z = b_1 \\
a_{21}x + a_{22}y + a_{23}z = b_2 \\
a_{31}x + a_{32}y + a_{33}z = b_3
\end{cases}
\iff
\begin{pmatrix}
a_{11} & a_{12} & a_{13} \\
a_{21} & a_{22} & a_{23} \\
a_{31} & a_{32} & a_{33}
\end{pmatrix}
\begin{pmatrix} x \\ y \\ z \end{pmatrix}
=
\begin{pmatrix} b_1 \\ b_2 \\ b_3 \end{pmatrix}
$$

La matriz ampliada se define como:

$$
(A \mid \mathbf{b}) = 
\left(\begin{array}{ccc|c}
a_{11} & a_{12} & a_{13} & b_1 \\
a_{21} & a_{22} & a_{23} & b_2 \\
a_{31} & a_{32} & a_{33} & b_3
\end{array}\right)
$$

Las **Operaciones Elementales por Fila (OEF)** preservan el conjunto solución del sistema:
1. **Tipo I (Intercambio):** $F_i \leftrightarrow F_j$.
2. **Tipo II (Escalamiento):** $F_i \to c \cdot F_i$ con $c \neq 0$.
3. **Tipo III (Combinación Lineal):** $F_i \to F_i + k \cdot F_j$.

> [!IMPORTANT]
> **Heurística USS de Mínimos Pasos en Gauss:**  
> Para evitar cálculos engorrosos con fracciones tempranas en certámenes:
> - Priorizar intercambios ($F_i \leftrightarrow F_j$) para ubicar pivotes con valor $\pm 1$.
> - Emplear combinaciones cruzadas enteras ($F_i \to a_{kk} F_i - a_{ik} F_k$).
> - Anular en bloque los elementos superiores e inferiores de la columna pivotante.

---

### 1.2 Teorema de Rouché-Frobenius y Clasificación Geométrica

| Caso | Condición de Rangos | Grados de Libertad | Interpretación Geométrica en $\mathbb{R}^3$ |
|:---|:---:|:---:|:---|
| **Compatible Determinado (SCD)** | $\operatorname{rg}(A) = \operatorname{rg}(A \mid \mathbf{b}) = 3$ | $0$ | Los $3$ planos se cortan en un **único punto** $P_0(x_0, y_0, z_0)$. |
| **Compatible Indeterminado (SCI)** | $\operatorname{rg}(A) = \operatorname{rg}(A \mid \mathbf{b}) = 2$ | $1$ | Los $3$ planos se cortan en una **recta común** $L(t)$. |
| **Compatible Indeterminado (SCI)** | $\operatorname{rg}(A) = \operatorname{rg}(A \mid \mathbf{b}) = 1$ | $2$ | Los $3$ planos son **coincidentes** en un único plano $\pi$. |
| **Incompatible (SI)** | $\operatorname{rg}(A) < \operatorname{rg}(A \mid \mathbf{b})$ | N/A | **Sin solución.** Planos paralelos disjuntos o prisma triangular hueco. |

In [ ]:
# Importaciones y configuración de visualización científica
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Paleta Institucional Universidad San Sebastián
USS_BLUE = '#00205B'
USS_GOLD = '#D4AF37'
USS_ACCENT_BLUE = '#1E88E5'
USS_ACCENT_GREEN = '#27AE60'
USS_ACCENT_RED = '#C0392B'
USS_DARK_GRAY = '#2C3E50'
USS_LIGHT_GRAY = '#F8F9F9'

# Configuración de renderizado matemático
sp.init_printing(use_latex='mathjax')
%matplotlib inline
print("Entorno USS configurado correctamente.")

In [ ]:
def analizar_rouche_frobenius(A_list, b_list):
    """Calcula rangos simbólicos exactos y clasifica según Rouché-Frobenius."""
    A_sp = sp.Matrix(A_list)
    b_sp = sp.Matrix(b_list)
    Ab_sp = A_sp.col_insert(3, b_sp)
    
    rg_A = A_sp.rank()
    rg_Ab = Ab_sp.rank()
    n = 3
    
    if rg_A == rg_Ab:
        if rg_A == n:
            tipo = "SCD"
            nombre = "Sistema Compatible Determinado"
            desc = "Solución ÚNICA: Intersección en un punto común P₀."
            gl = 0
        else:
            tipo = "SCI"
            nombre = "Sistema Compatible Indeterminado"
            gl = n - rg_A
            desc = f"INFINITAS SOLUCIONES ({gl} grado(s) de libertad)."
    else:
        tipo = "SI"
        nombre = "Sistema Incompatible"
        desc = "SIN SOLUCIÓN: Los planos no tienen ningún punto común simultáneo."
        gl = None
        
    return {
        "A_sp": A_sp,
        "b_sp": b_sp,
        "Ab_sp": Ab_sp,
        "rg_A": rg_A,
        "rg_Ab": rg_Ab,
        "tipo": tipo,
        "nombre": nombre,
        "descripcion": desc,
        "grados_libertad": gl
    }

def resolver_gauss_jordan_pasos(A_sp, b_sp):
    """Genera la reducción por filas a forma escalonada reducida (RREF)."""
    Ab = A_sp.col_insert(3, b_sp)
    rref_mat, pivot_cols = Ab.rref()
    
    symbols = sp.symbols('x y z')
    sol = sp.linsolve((A_sp, b_sp), symbols)
    
    return Ab, rref_mat, pivot_cols, sol

In [ ]:
def graficar_sistema_3d(A_np, b_np, rf_info, elev=25, azim=45):
    """Grafica los 3 planos en R³ con alta fidelidad y anotación de soluciones."""
    fig = plt.figure(figsize=(10, 8), facecolor='white')
    ax = fig.add_subplot(111, projection='3d')
    
    x_vals = np.linspace(-4, 4, 30)
    y_vals = np.linspace(-4, 4, 30)
    X, Y = np.meshgrid(x_vals, y_vals)
    
    colores = [USS_BLUE, USS_GOLD, USS_ACCENT_BLUE]
    etiquetas = [
        f"π₁: {A_np[0,0]:.0f}x + {A_np[0,1]:.0f}y + {A_np[0,2]:.0f}z = {b_np[0]:.0f}",
        f"π₂: {A_np[1,0]:.0f}x + {A_np[1,1]:.0f}y + {A_np[1,2]:.0f}z = {b_np[1]:.0f}",
        f"π₃: {A_np[2,0]:.0f}x + {A_np[2,1]:.0f}y + {A_np[2,2]:.0f}z = {b_np[2]:.0f}"
    ]
    
    for i in range(3):
        a, b_coef, c = A_np[i]
        d = b_np[i]
        
        if abs(c) >= 1e-4:
            Z = (d - a * X - b_coef * Y) / c
            ax.plot_surface(X, Y, Z, color=colores[i], alpha=0.45, edgecolor='none')
        elif abs(b_coef) >= 1e-4:
            z_vals = np.linspace(-4, 4, 30)
            X_p, Z_p = np.meshgrid(x_vals, z_vals)
            Y_p = (d - a * X_p - c * Z_p) / b_coef
            ax.plot_surface(X_p, Y_p, Z_p, color=colores[i], alpha=0.45, edgecolor='none')
        elif abs(a) >= 1e-4:
            z_vals = np.linspace(-4, 4, 30)
            Y_p, Z_p = np.meshgrid(y_vals, z_vals)
            X_p = (d - b_coef * Y_p - c * Z_p) / a
            ax.plot_surface(X_p, Y_p, Z_p, color=colores[i], alpha=0.45, edgecolor='none')

    # Marcadores de solución según tipo
    if rf_info["tipo"] == "SCD":
        try:
            sol = np.linalg.solve(A_np, b_np)
            ax.scatter([sol[0]], [sol[1]], [sol[2]], color=USS_ACCENT_RED, s=200, zorder=10,
                       edgecolor='black', linewidth=1.5, label=f"Solución P₀({sol[0]:.2f}, {sol[1]:.2f}, {sol[2]:.2f})")
            ax.text(sol[0] + 0.2, sol[1] + 0.2, sol[2] + 0.2,
                    f"P₀({sol[0]:.2f}, {sol[1]:.2f}, {sol[2]:.2f})",
                    color=USS_ACCENT_RED, fontweight='bold', fontsize=11)
        except Exception:
            pass
    elif rf_info["tipo"] == "SCI" and rf_info["grados_libertad"] == 1:
        # Trazar recta paramétrica aproximada
        try:
            t = np.linspace(-3, 3, 50)
            # Solución de mínimos cuadrados para punto base y vector director
            p_base = np.linalg.lstsq(A_np, b_np, rcond=None)[0]
            _, _, vh = np.linalg.svd(A_np)
            v_dir = vh[-1]
            line_pts = p_base[:, None] + v_dir[:, None] * t[None, :]
            ax.plot(line_pts[0], line_pts[1], line_pts[2], color=USS_ACCENT_RED, linewidth=3.5, label="Recta de Solución L")
        except Exception:
            pass

    ax.set_title(f"{rf_info['nombre']} ({rf_info['tipo']})\n" f"rg(A) = {rf_info['rg_A']} | rg(A|b) = {rf_info['rg_Ab']} | n = 3",
                 fontsize=12, fontweight='bold', color=USS_BLUE, pad=15)
    ax.set_xlabel('Eje X', fontweight='bold', color=USS_DARK_GRAY)
    ax.set_ylabel('Eje Y', fontweight='bold', color=USS_DARK_GRAY)
    ax.set_zlabel('Eje Z', fontweight='bold', color=USS_DARK_GRAY)
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax.set_zlim(-4, 4)
    ax.view_init(elev=elev, azim=azim)
    ax.grid(True, linestyle=':', alpha=0.6)
    
    # Crear leyenda limpia
    p1 = plt.Rectangle((0, 0), 1, 1, fc=USS_BLUE, alpha=0.5)
    p2 = plt.Rectangle((0, 0), 1, 1, fc=USS_GOLD, alpha=0.5)
    p3 = plt.Rectangle((0, 0), 1, 1, fc=USS_ACCENT_BLUE, alpha=0.5)
    ax.legend([p1, p2, p3], etiquetas, loc='upper left', fontsize=9)
    plt.tight_layout()
    plt.show()

In [ ]:
# Selector interactivo de casos predefinidos y ángulos 3D
sistemas_predefinidos = {
    "1. SCD (Solución Única P₀)": {
        "A": [[2, 1, -1], [-3, -1, 2], [-2, 1, 2]],
        "b": [8, -11, -3]
    },
    "2. SCI (Infinitas Soluciones — Recta Común)": {
        "A": [[1, 1, 1], [2, -1, 3], [3, 0, 4]],
        "b": [3, 4, 7]
    },
    "3. SI (Incompatible — Prisma Triangular Hueco)": {
        "A": [[1, 1, 1], [1, -1, 2], [2, 0, 3]],
        "b": [1, 2, 5]
    }
}

dropdown_caso = widgets.Dropdown(
    options=list(sistemas_predefinidos.keys()),
    value=list(sistemas_predefinidos.keys())[0],
    description="Sistema:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='60%')
)

slider_elev = widgets.IntSlider(value=25, min=0, max=90, step=5, description="Elevación:")
slider_azim = widgets.IntSlider(value=45, min=0, max=360, step=5, description="Azimut:")

out_display = widgets.Output()

def actualizar_simulador(change=None):
    with out_display:
        clear_output(wait=True)
        caso_sel = sistemas_predefinidos[dropdown_caso.value]
        A_val = caso_sel["A"]
        b_val = caso_sel["b"]
        
        rf = analizar_rouche_frobenius(A_val, b_val)
        Ab_init, rref_mat, pivotes, sol = resolver_gauss_jordan_pasos(rf["A_sp"], rf["b_sp"])
        
        display(HTML(f"""
        <div style="background-color: {USS_LIGHT_GRAY}; padding: 12px; border-left: 5px solid {USS_BLUE}; border-radius: 4px; margin-bottom: 12px;">
            <h3 style="color: {USS_BLUE}; margin: 0;">Diagnóstico: {rf['nombre']} ({rf['tipo']})</h3>
            <p style="margin: 4px 0 0 0; color: {USS_DARK_GRAY}; font-size: 14px;">
                <b>Condición de Rangos:</b> rg(A) = <b>{rf['rg_A']}</b> | rg(A|b) = <b>{rf['rg_Ab']}</b> | n = 3<br>
                <b>Interpretación:</b> {rf['descripcion']}
            </p>
        </div>
        """))
        
        print("Matriz Ampliada Inicial (A|b):")
        display(Ab_init)
        
        print("Forma Escalonada Reducida por Filas (RREF):")
        display(rref_mat)
        
        print(f"Conjunto Solución Analítico: {sol}")
        
        graficar_sistema_3d(np.array(A_val, dtype=float), np.array(b_val, dtype=float), rf,
                            elev=slider_elev.value, azim=slider_azim.value)

dropdown_caso.observe(actualizar_simulador, names='value')
slider_elev.observe(actualizar_simulador, names='value')
slider_azim.observe(actualizar_simulador, names='value')

display(widgets.VBox([
    dropdown_caso,
    widgets.HBox([slider_elev, slider_azim]),
    out_display
]))

actualizar_simulador()

---

## 🎯 Síntesis Táctica para Evaluaciones (Solemnes USS)

> [!EXAMPLE]
> **Checklist Rápido de Rouché-Frobenius:**
> 1. **Triángulo de Ceros Completo:** Si tras aplicar OEF obtienes 3 pivotes distintos de cero en la diagonal principal, el sistema es **SCD** de forma inmediata.
> 2. **Fila Nula Absoluta $(0 \ 0 \ 0 \mid 0)$:** Significa que una de las ecuaciones era una combinación lineal de las otras dos (redundancia). Si no hay contradicciones, el sistema es **SCI**.
> 3. **Fila Contradictoria $(0 \ 0 \ 0 \mid c)$ con $c \neq 0$:** Representa la proposición falsa $0 = c$. En ese instante se detiene la eliminación Gaussiana y se concluye **SI** (Sistema Incompatible).